In [102]:
import pandas as pd
import os
import re
from mistralai.client import Mistral
from rapidfuzz import fuzz, process
from tqdm import tqdm
from dotenv import load_dotenv
import json

In [56]:
load_dotenv(dotenv_path='../.env')

True

In [57]:
client = Mistral(api_key=os.getenv('MISTRAL_API_KEY'))

In [58]:
TEMPLATE_OLD_PATH = "../data/submission_template.csv"
OLD_SUBMISSION = "./submission2.csv"

In [59]:
old_submission_df = pd.read_csv(OLD_SUBMISSION)
template_old_df = pd.read_csv(TEMPLATE_OLD_PATH)
old_submission_df['doc_id'] = template_old_df['doc_id']
old_submission_df['party_name'] = template_old_df['party_name']
old_submission_df

,id,votes,doc_id,party_name
0,constituency_10_1_1,14813,constituency_10_1,ประชาธิปัตย์
1,constituency_10_1_2,14368,constituency_10_1,ภูมิใจไทย
2,constituency_10_1_3,979,constituency_10_1,เศรษฐกิจ
3,constituency_10_1_4,244,constituency_10_1,กล้าธรรม
4,constituency_10_1_5,351,constituency_10_1,พลวัต
...,...,...,...,...
10048,party_list_34_11_53,14,party_list_34_11,ไทยพิทักษ์ธรรม
10049,party_list_34_11_54,41,party_list_34_11,ความหวังใหม่
10050,party_list_34_11_55,29,party_list_34_11,ไทยรวมไทย
10051,party_list_34_11_56,41,party_list_34_11,เพื่อบ้านเมือง


#### Extraction

In [86]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [120]:
KNOWN_PARTIES = [
    'ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
    'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
    'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
    'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
    'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
    'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
    'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
    'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
    'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ',
    'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
    'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
    'สร้างชาติ', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
    'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
    'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
    'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธรรมใหม่', 'กรีน',
    'แผ่นดินธรรม', 'ประชาไทย', 'ประชาอาสาชาติ',
    'เครือข่ายชาวนาแห่งประเทศไทย', 'ไทยรวมไทย', 'พลังไทยรักชาติ', 'ใหม่'
]

def is_party(text, threshold=70):
    match = process.extractOne(text, KNOWN_PARTIES, scorer=fuzz.ratio)
    return match and match[1] >= threshold

In [91]:
def extract_party_score_dict(md: str) -> dict:
    result = {}

    for line in md.split('\n'):
        if '---' in line or 'รวมคะแนน' in line:
            continue

        parts = [p.strip() for p in line.split('|') if p.strip()]
        if len(parts) < 2:
            continue

        for i, p in enumerate(parts):
            match = process.extractOne(p, KNOWN_PARTIES, scorer=fuzz.ratio)
            if match and match[1] >= 60:
                canonical_party = match[0]
                for vote_raw in reversed(parts[i+1:]):
                    try:
                        vote = thai_num_to_int(vote_raw)
                        result[canonical_party] = vote
                        break
                    except Exception:
                        pass
                break

    return result

In [ ]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        filename = os.path.basename(path)
        with open(path, "rb") as f:
            uploaded = client.files.upload(
                file={"file_name": filename, "content": f},
                purpose="ocr"
            )
        signed_url = client.files.get_signed_url(file_id=uploaded.id)
        result = client.ocr.process(
            model="mistral-ocr-latest",
            document={"type": "document_url", "document_url": signed_url.url}
        )
        # Combine all pages markdown
        #print(result.pages[0].markdown)
        md = "".join(page.markdown for page in result.pages)
        return extract_party_score_dict(md)
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [89]:
def assign_votes(df, vote_dict, threshold=60):
    vote_dict = {k: v for k, v in vote_dict.items() if k != 'รวมคะแนนทั้งสิ้น'}
    keys = list(vote_dict.keys())

    def get_vote(party_name):
        if pd.isna(party_name):
            return 0
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.token_set_ratio
        )
        if match is None:
            return 0
        best_key, score, _ = match
        return vote_dict[best_key] if score >= threshold else 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

### Run

In [65]:
submission_df = old_submission_df.copy()

In [96]:
bad_one = [
	'constituency_12_6',
	'party_list_10_24',
	'party_list_10_31',
	'party_list_10_9',
	'party_list_14_2',
	'party_list_19_2',
	'party_list_20_1',
	'party_list_20_5',
	'party_list_20_7',
	'party_list_21_5',
	'party_list_22_2',
	'party_list_30_2',
	'party_list_30_3',
	'party_list_30_4',
	'party_list_31_8',
	'party_list_32_7',
	'party_list_33_3',
	'party_list_33_6'
]

In [70]:
PREFIX = "../pdf/"

In [97]:
happen = {}

for doc_id in tqdm(bad_one, desc="Processing", unit="doc"):
	try:
		vote_dict = extraction(PREFIX + doc_id + '.pdf')
		happen[doc_id] = vote_dict
		mask = submission_df["doc_id"] == doc_id
		submission_df.loc[mask] = assign_votes(
			submission_df.loc[mask].copy(),
			vote_dict,
			threshold=60
		)
	except Exception as e:
		print(f"{doc_id}: {str(e)}")

Processing:   6%|▌         | 1/18 [00:03<01:06,  3.89s/doc]

|  หมายเลข
ประจำตัว
ผู้สมัคร | ชื่อตัว – ชื่อสกุล
ผู้สมัครรับเลือกตั้ง | สังกัด
พรรค
การเมือง | ได้คะแนน
(ให้กรอกทั้งตัวเลขและตัวอักษร)  |
| --- | --- | --- | --- |
|  ๔ | นายสุทัศน์ มิศิริ | ประชาชน | ๔๔,๔๐๕ (สี่หมื่นสี่พันสี่ร้อยห้า)  |
|  ๒ | นางสาวศศิประภา ดีรอด | ภูมิใจไทย | ๒๕,๒๑๘ (สองหมื่นห้าพันสองร้อยสิบแปด)  |
|  ๕ | นายประมการ อ่วมอ่อง | เพื่อไทย | ๑๗,๙๖๘ (หนึ่งหมื่นเจ็ดพันเก้าร้อยหกสิบแปด)  |
|  ๑ | นายพงศกร นกทรัพย์ | ประชาธิปไตย | ๖,๕๙๕ (หกพันห้าร้อยเก้าสิบห้า)  |
|  ๘ | นายเกริกหิรัญ แปลกประเสริฐ | เศรษฐกิจ | ๑,๗๔๑ (หนึ่งพันเจ็ดร้อยสี่สิบเอ็ด)  |
|  ๙ | นายชานนท์ พงษ์เจริญ | ทางเลือกใหม่ | ๑,๓๘๙ (หนึ่งพันสามร้อยแปดสิบเก้า)  |
|  ๑๑ | นายธีรวัจน์ ดารามาศ | พลวัต | ๘๘๗ (แปดร้อยแปดสิบเจ็ด)  |
|  ๑๒ | นายคุณากร มั่นนทีรัย | กล้าธรรม | ๘๒๓ (แปดร้อยยี่สิบสาม)  |
|  รวมคะแนนทั้งสิ้น |   |   | ๙๙,๐๒๖ (เก้าหมื่นเก้าพันยี่สิบหก)  |


Processing:  11%|█         | 2/18 [00:11<01:40,  6.29s/doc]

|  **ความเคยจะเป็นข้าวหรือของพวกเขาวะมือ>** | **ข้อพรรคการมือ>** | **1. ให้พรรคการ (ให้พวกเขาได้รับประเภทสำหรับพรรค)**  |
| --- | --- | --- |
|  ๑ | โพดทรัพย์ทวี | ส่ง: (สำหรับผลิต)  |
|  ๒ | เตือกปลื้อก | แบบ: (หนึ่งในหากสักจะลด)  |
|  ๓ | ไหม | แบบ: (นกสีจะมือ)  |
|  ๔ | อิชั่วทอ | แบบ: (หนึ่งในหากสักจะลด)  |
|  ๕ | รวมไออิคม | แบบ: (สามในหลักสาม)  |
|  ๖ | รวมไออิสร้างขาด | แบบ: (หนึ่งในหลักสาม)  |
|  ๗ | ขยะใย | แบบ: (หนึ่งในจะกำลังงดง)  |
|  ๘ | ประชาสิ่งใหม่ใหม่ | แบบ: (หนึ่งในจะกำลังง)  |
|  ๙ | เพื่อไทย | แบบ: (เพื่อขับสารที่สะดวกสลับงดง)  |
|  ๑+ | ชาวเมืองไทย | แบบ: (สามในหนึ่งหรือสี่)  |
|  ๒+ | เหรอภูมิอ | แบบ: (หนึ่งในเพื่อให้พวกเขามีความรู้)  |
|  ๓+ | เสร็จวนไทย | แบบ: (สำคัญหนึ่ง)  |
|  ๔+ | รวมพลังประชาชน | แบบ: (หนึ่งในจะได้รับประเภท)  |
|  ๕+ | ห้องซิโคม | แบบ: (สำคัญ)  |
|  ๖+ | อะเหงน้อย | ส่ง: (สำคัญ)  |
|  ๗+ | หลังเพื่อไทย | ส่ง: (สำคัญ)  |
|  **ความเคยจะเป็นข้าวหรือของพวกเขาวะมือ>** | **ข้อพรรคการมือ>** | **1. ให้พรรคการ (ให้พวกเขาได้รับประเภทสำหรับพรรค)**  |
|

Processing:  17%|█▋        | 3/18 [00:21<01:57,  7.83s/doc]

...

|  ขอขอบเขต
ของบัญชีการเชื่อ
ของพรรค
การเมือง | ชื่อ
พรรคการเมือง | โต้ตอบแนว
(ให้การถกเยี่ยงตัวเลขและตัวอักษร) | หมายเหตุ  |
| --- | --- | --- | --- |
|  ๑ | ไทยทรัพย์ทวี | ๔๖๖ (สิร้อยสีสันพก) |   |
|  ๒ | เพื่อชาติไทย | ๑,๒๒๕ (หนึ่งพันสองร้อยสีสันเท้า) |   |
|  ๓ | ไหม | ๑,๘๕๗ (หนึ่งพันแปดร้อยสีเส้นเอ็ด) |   |
|  ขอขอบเขต
ของบัญชีการเชื่อ
ของพรรค
การเมือง | ชื่อ
พรรคการเมือง | โต้ตอบแนว
(ให้การถกเยี่ยงตัวเลขและตัวอักษร) | หมายเหตุ  |
|  ๔ | มีดีไหม | ๒,๓๔๔ (สองพันสามร้อยสีสันสี) |   |
|  ๕ | รวมใจไทย | ๕๗๑ (ห้าร้อยเก้าสิบเอ็ด) |   |
|  ๖ | รวมไทยสร้างชาติ | ๑,๑๕๖ (หนึ่งพันหนึ่งร้อยสีเส้นพก) |   |
|  ๗ | พลวัต | ๑๐๗ (หนึ่งร้อยเก้า) |   |
|  ๘ | ประชาธิปไตยไหม | ๑๐๑ (สิร้อยสาม) |   |
|  ๙ | เพื่อไทย | ๒๐,๓๘๙ (สองพันสามร้อยแปดสิบเก้า) |   |
|  ๑๐ | ทางเลือกไหม | ๓๓๖ (สามร้อยสามสันพก) |   |
|  ๑๑ | เศรษฐกิจ | ๕,๓๑๐ (ห้าพันสามร้อยสีบ) |   |
|  ๑๒ | เสรีรวมไทย | ๓๓๔ (สามร้อยสามสีบสี) |   |
|  ๑๓ | รวมพลังประชาชน | ๓๕๒ (สามร้อยสีเส้นสอง) |   |
|  ๑๔ | ท้องที่ไทย | ๕๖ (ห้าสิบพก) |   |
|  ๑๕ 

Processing:  94%|█████████▍| 17/18 [02:45<00:09,  9.92s/doc]

|  หมายเลขของบัญชี
รายชื่อของพรรค
การเมือง | สังกัด
พรรคการเมือง | ได้คะแนน
(ให้กรอกทั้งตัวเลขและตัวอักษร)  |
| --- | --- | --- |
|  ๑ | ไทยทรัพย์ทวี | ๓๙๓ (สามร้อยเก้าสิบสาม)  |
|  ๒ | เพื่อชาติไทย | ๑,๕๙๑ (หนึ่งพันห้าร้อยเก้าสิบเอ็ด)  |
|  ๓ | ไน่น่ | ๔๕๗ (สี่ร้อยห้าสิบเจ็ด)  |
|  ๔ | มิติไน่น่ | ๒,๙๕๘ (สองพันเก้าร้อยห้าสิบแปด)  |
|  ๕ | รวมใจไทย | ๑,๔๙๐ (หนึ่งพันสี่ร้อยเก้าสิบ)  |
|  ๖ | รวมไทยสร้างชาติ | ๘๑๖ (แปดร้อยสิบหก)  |
|  ๗ | พลวัต | ๓๔๔ (สามร้อยสี่สิบสี่)  |
|  ๘ | ประชาธิปไตยไน่น่ | ๕๑๐ (ห้าร้อยสิบ)  |
|  ๙ | เพื่อไทย | ๑๗,๖๙๗ (หนึ่งหมื่นเจ็ดพันหกร้อยเก้าสิบเจ็ด)  |
|  ๑๐ | ทางเลือกไน่น่ | ๓๒๙ (สามร้อยยี่สิบเก้า)  |
|  ๑๑ | เศรษฐกิจ | ๔,๐๔๔ (สี่พันสี่สิบสี่)  |
|  ๑๒ | เสรีรวมไทย | ๓๙๑ (สามร้อยเก้าสิบเอ็ด)  |
|  ๑๓ | รวมพลังประชาชน | ๔๘๑ (สี่ร้อยแปดสิบเอ็ด)  |
|  ๑๔ | ท้องที่ไทย | ๑๐๙ (หนึ่งร้อยเก้า)  |
|  ๑๕ | อนาคตไทย | ๘๙ (แปดสิบเก้า)  |
|  ๑๖ | พลังเพื่อไทย | ๒๕๕ (สองร้อยห้าสิบห้า)  |
|  ๑๗ | ไทยชนะ | ๑๖๘ (หนึ่งร้อยหกสิบแปด)  |
|  ๑๘ | พลังสังคมไน่น่ | ๓๑ (สามสิบเอ็ด) 

Processing: 100%|██████████| 18/18 [02:56<00:00,  9.82s/doc]

|  ขอบเขต
ของบัญชีรายชื่อ
ของสภาค
การเมือง | ชื่อ
พรรคการเมือง | ได้คะแนน
(ให้การถกที่สร้างลานลวดตัวอักษร) | หมายเหตุ  |
| --- | --- | --- | --- |
|  ๑ | ไทยทรัพย์ทวี | ๕๖๘ (ห้าร้อยสามสิบแปด) |   |
|  ๒ | เพื่อชาติไทย | ๓,๓๑๐ (สามพันสามร้อยสิบ) |   |
|  ขอบเขต
ของบัญชีรายชื่อ
ของสภาค
การเมือง | ชื่อ
พรรคการเมือง | ได้คะแนน
(ให้การถกที่สร้างลานลวดตัวอักษร) | หมายเหตุ  |
|  ๓ | ไทม์ | ๒๗๔ (สองร้อยเจ็ดสิบเก้า) |   |
|  ๔ | มีติไทม์ | ๒,๒๑๓ (สองพันสองร้อยสิบสาม) |   |
|  ๕ | รวมใจไทย | ๑๖๔ (สี่ร้อยสามสิบสี่) |   |
|  ๖ | รวมไทยสร้างชาติ | ๘๔๘ (แปดร้อยเก้าสิบแปด) |   |
|  ๗ | พลวัต | ๑๒๖ (หนึ่งร้อยสี่สิบหก) |   |
|  ๘ | ประชาธิปไตยไทม์ | ๗๐๓ (เจ็ดร้อยสาม) |   |
|  ๙ | เพื่อไทย | ๒๓,๓๕๗ (สองหมื่นสามพันสามร้อยห้าสิบเจ็ด) |   |
|  ๑๐ | ทางเลือกไทม์ | ๒๕๘ (สองร้อยห้าสิบแปด) |   |
|  ๑๑ | เศรษฐกิจ | ๒,๒๐๔ (สองพันสองร้อยเก้า) |   |
|  ๑๒ | เสร็จรวมไทย | ๒๒๗ (สองร้อยสี่สิบเจ็ด) |   |
|  ๑๓ | รวมพลังประชาชน | ๓๖๓ (สามร้อยหกสิบสาม) |   |
|  ๑๔ | ท้องที่ไทย | ๕๕ (ห้าสิบห้า) |   |
|  ๑๕ | อนาคตไทย | ๕

In [98]:
output_df = submission_df.drop(['doc_id', 'party_name'], axis=1)
output_df

,id,votes
0,constituency_10_1_1,14813
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,29
10051,party_list_34_11_56,41


In [99]:
output_df.to_csv("submission3.csv", index=False)

In [100]:
for key, values in happen.items():
	if values == {}:
		print(key)

party_list_10_24


In [121]:
with open("../anormaly/party_list_10_24.json") as f:
    data = json.load(f)

for item in data:
    party = item["สังกัดพรรคการเมือง"].strip()
    vote_raw = item["คะแนน"].strip()
    try:
        vote = thai_num_to_int(vote_raw)
        match = process.extractOne(party, KNOWN_PARTIES, scorer=fuzz.ratio)
        if match and match[1] >= 60:
            canonical = match[0]
            mask = (submission_df["doc_id"] == "party_list_10_24") & (submission_df["party_name"] == canonical)
            submission_df.loc[mask, "votes"] = vote
    except Exception as e:
        print(f"Error: {party} - {e}")

In [123]:
output_df = submission_df.drop(['doc_id', 'party_name'], axis=1)
output_df

,id,votes
0,constituency_10_1_1,14813
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,29
10051,party_list_34_11_56,41


In [124]:
output_df.to_csv("submission3.csv", index=False)